# Buổi 6 - Pandas nâng cao II: Data Cleaning, GroupBy và Pivot Table

Notebook này bám sát bài giảng Buổi 6 và được viết lại theo hướng **chạy lại được, giải thích được**. Nội dung gồm:

1. Làm sạch dữ liệu: trùng lặp, ánh xạ, sentinel và đổi tên.
2. Phân nhóm số liên tục, xử lý outlier và mã hóa biến phân loại.
3. `GroupBy` theo mô hình Split-Apply-Combine.
4. `pivot_table()` và các bài áp dụng trên Planets, Titanic, CDC Births, MovieLens 1M.

> Hãy chạy notebook từ trên xuống dưới trong kernel mới. Các cell kiểm tra dùng `assert` để phát hiện sớm lỗi đường dẫn hoặc kết quả sai.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

def find_repo_root(start: Path) -> Path:
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / "slides" / "buoi6_python_datascience.pdf").exists():
            return candidate
    raise FileNotFoundError("Không tìm thấy thư mục gốc của repo.")

REPO_ROOT = find_repo_root(Path.cwd().resolve())
DATA_DIR = REPO_ROOT / "datasets" / "buoi6"
MOVIELENS_DIR = DATA_DIR / "movielens"

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)
print(f"Pandas: {pd.__version__}")
print(f"Dữ liệu: {DATA_DIR}")

Pandas: 2.3.3
Dữ liệu: C:\Hon\Nam_3_HK1 2026_2027\DataScience\Hoan-Data-Science-Course\datasets\buoi6


## 1. Làm sạch dữ liệu I - trùng lặp và thay thế giá trị

**Mục đích:** chuẩn hóa dữ liệu ở mức từng dòng trước khi tổng hợp. Một dòng trùng lặp có thể làm sai tổng, trung bình và số đếm; một mã sentinel như `-999` có thể bị hiểu nhầm là giá trị thật.

In [2]:
duplicate_data = pd.DataFrame({
    "k1": ["one", "two"] * 3 + ["two"],
    "k2": [1, 1, 2, 3, 3, 4, 4],
})

original_duplicate_count = int(duplicate_data.duplicated().sum())
print(f"Dòng trùng hoàn toàn: {original_duplicate_count}")
display(duplicate_data.assign(is_duplicate=duplicate_data.duplicated()))

print("Bỏ trùng hoàn toàn, giữ lần xuất hiện đầu:")
display(duplicate_data.drop_duplicates())

duplicate_data["v1"] = range(len(duplicate_data))
print("Chỉ xét trùng theo k1 và giữ dòng cuối:")
display(duplicate_data.drop_duplicates(subset=["k1"], keep="last"))

Dòng trùng hoàn toàn: 1


,k1,k2,is_duplicate
0,one,1,False
1,two,1,False
2,one,2,False
3,two,3,False
4,one,3,False
5,two,4,False
6,two,4,True


Bỏ trùng hoàn toàn, giữ lần xuất hiện đầu:


,k1,k2
0,one,1
1,two,1
2,one,2
3,two,3
4,one,3
5,two,4


Chỉ xét trùng theo k1 và giữ dòng cuối:


,k1,k2,v1
4,one,3,4
6,two,4,6


### Điểm quan trọng về dữ liệu trùng

- `duplicated()` đánh dấu dòng lặp lại; `drop_duplicates()` loại dòng đó.
- `subset` phải phản ánh **khóa nghiệp vụ**. Hai người cùng tuổi không có nghĩa là cùng một người.
- `keep='first'`, `'last'` hoặc `False` quyết định bản ghi nào được giữ. Không nên chọn tùy tiện nếu các bản ghi có thời điểm cập nhật khác nhau.

In [ ]:
food_data = pd.DataFrame({
    "food": ["bacon", "pulled pork", "pastrami", "corned beef", "honey ham", "nova lox"],
    "ounces": [4, 3, 6, 7.5, 5, 6],
})
meat_to_animal = {
    "bacon": "pig", "pulled pork": "pig", "pastrami": "cow",
    "corned beef": "cow", "honey ham": "pig", "nova lox": "salmon",
}
food_data["animal"] = food_data["food"].map(meat_to_animal)
display(food_data)

sentinel_values = pd.Series([1.0, -999.0, 2.0, -999.0, -1000.0, 3.0], name="raw_value")
clean_values = sentinel_values.replace({-999.0: np.nan, -1000.0: 0.0})
display(pd.concat([sentinel_values, clean_values.rename("clean_value")], axis=1))

labels = pd.DataFrame(
    np.arange(12).reshape(3, 4),
    index=["Ohio", "Colorado", "New York"],
    columns=["one", "two", "three", "four"],
)
renamed = labels.rename(index={"Ohio": "INDIANA"}, columns={"three": "peekaboo"})
display(renamed)

`map()` phù hợp khi tạo cột mới từ một quy tắc ánh xạ. Giá trị không có trong dictionary sẽ thành `NaN`, vì vậy phải kiểm tra sau khi ánh xạ. `replace()` thay trực tiếp các giá trị đã biết, còn `rename()` đổi nhãn hàng/cột và mặc định không sửa DataFrame gốc.

In [ ]:
assert original_duplicate_count == 1
assert duplicate_data.drop_duplicates().shape[0] == 7  # v1 làm mỗi dòng trở nên duy nhất
assert food_data["animal"].notna().all(), "Dictionary ánh xạ còn thiếu giá trị."
assert clean_values.isna().sum() == 2
assert "three" in labels.columns and "peekaboo" in renamed.columns
print("✓ Module 21 đạt các kiểm tra cơ bản.")

## 2. Làm sạch dữ liệu II - phân khoảng, outlier và mã hóa

**Mục đích:** chuyển số liên tục thành nhóm dễ tổng hợp, giảm ảnh hưởng cực đoan khi có căn cứ, và chuyển biến phân loại thành dạng số mà thuật toán có thể sử dụng.

In [ ]:
ages = pd.Series([20, 22, 25, 27, 21, 23, 37, 31, 61, 45, 41, 32], name="age")
age_bins = [18, 25, 35, 60, 100]
age_labels = ["Youth", "YoungAdult", "MiddleAged", "Senior"]
age_group = pd.cut(ages, bins=age_bins, labels=age_labels, include_lowest=True)
display(pd.DataFrame({"age": ages, "age_group": age_group}))

rng = np.random.default_rng(12345)
sample = pd.Series(rng.standard_normal(1000), name="value")
quartile = pd.qcut(sample, q=4, precision=2)
print("Số quan sát trong từng nhóm qcut:")
display(quartile.value_counts(sort=False))

### `cut()` và `qcut()` khác nhau thế nào?

- `cut()` dùng ranh giới do người phân tích chọn. Nó phù hợp khi ranh giới có ý nghĩa nghiệp vụ, ví dụ nhóm tuổi.
- `qcut()` tự chọn ranh giới theo phân vị để số quan sát mỗi nhóm gần bằng nhau. Nó phù hợp khi muốn so sánh các nhóm có kích thước cân bằng.
- Phải để ý quy ước đóng/mở khoảng. Mặc định `(18, 25]` không chứa 18 nhưng có chứa 25; `include_lowest=True` giúp nhận cả giá trị thấp nhất ở khoảng đầu.

In [ ]:
rng = np.random.default_rng(12345)
outlier_data = pd.DataFrame(rng.standard_normal((1000, 4)), columns=list("ABCD"))
outlier_mask = outlier_data.abs() > 3
rows_with_outlier = outlier_mask.any(axis="columns")
print(f"Số ô vượt |3|: {int(outlier_mask.sum().sum())}")
print(f"Số dòng có ít nhất một giá trị vượt |3|: {int(rows_with_outlier.sum())}")

capped_data = outlier_data.clip(lower=-3, upper=3)
display(capped_data.describe().round(3))
assert capped_data.abs().max().max() <= 3


**Không phải giá trị lớn nào cũng là lỗi.** Ngưỡng `|x| > 3` chỉ hợp lý khi dữ liệu đã ở thang đo phù hợp và giả định phân bố có căn cứ. `clip()` (capping) giữ nguyên số dòng nhưng làm mất độ lớn thật ở hai đuôi; chỉ dùng sau khi kiểm tra nguồn dữ liệu và mục tiêu phân tích.

In [ ]:
category_data = pd.DataFrame({"key": ["b", "b", "a", "c", "a", "b"], "data1": range(6)})
category_dummies = pd.get_dummies(category_data["key"], prefix="key", dtype=int)
display(category_data.join(category_dummies))

movies_path = MOVIELENS_DIR / "movies.dat"
movie_columns = ["movie_id", "title", "genres"]
movies = pd.read_table(
    movies_path, sep="::", header=None, names=movie_columns,
    engine="python", encoding="latin-1",
)
genre_dummies = movies["genres"].str.get_dummies("|")
print(f"MovieLens movies: {movies.shape}; ma trận thể loại: {genre_dummies.shape}")
display(pd.concat([movies.head(3), genre_dummies.head(3)], axis=1))

`pd.get_dummies()` dùng cho một nhãn trên mỗi dòng. Riêng cột `genres`, một phim có thể thuộc nhiều thể loại ngăn cách bằng `|`, nên dùng `Series.str.get_dummies('|')`. Mỗi cột kết quả trả lời câu hỏi: “phim này có thuộc thể loại đó không?”.

In [ ]:
assert age_group.notna().all()
assert quartile.value_counts().eq(250).all()
assert category_dummies.sum(axis=1).eq(1).all()
assert movies.shape == (3883, 3)
assert genre_dummies.shape[1] == 18
print("✓ Module 22 đạt các kiểm tra cơ bản.")

## 3. GroupBy - Split-Apply-Combine

`groupby()` chưa tính toán ngay. Nó mô tả cách **Split** dữ liệu theo khóa; một hàm sau đó **Apply** trên từng nhóm; Pandas cuối cùng **Combine** các kết quả. Chọn đúng loại thao tác giúp code rõ và nhanh hơn.

In [ ]:
group_data = pd.DataFrame({
    "key": ["A", "B", "C", "A", "B", "C"],
    "data1": range(6),
    "data2": np.random.default_rng(0).integers(0, 10, 6),
})
grouped = group_data.groupby("key")
print(type(grouped))
display(grouped[["data1", "data2"]].sum())

In [ ]:
summary = grouped.agg(
    data1_min=("data1", "min"),
    data1_median=("data1", "median"),
    data2_max=("data2", "max"),
)
display(summary)

filtered = grouped.filter(lambda frame: frame["data2"].std() > 2)
display(filtered)

centered = grouped[["data1", "data2"]].transform(lambda column: column - column.mean())
display(centered)

- `agg()` thu gọn mỗi nhóm thành một hoặc vài thống kê.
- `filter()` giữ hoặc loại **toàn bộ nhóm** theo một điều kiện.
- `transform()` trả về cùng số dòng với dữ liệu gốc, thích hợp để chuẩn hóa theo nhóm.
- `apply()` linh hoạt nhất nhưng thường chậm hơn; ưu tiên hàm chuyên dụng khi có thể.

In [ ]:
def add_data1_share(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    total = result["data1"].sum()
    result["data1_share"] = result["data1"] / total if total else np.nan
    return result

applied = (
    group_data.groupby("key", group_keys=False)[["data1", "data2"]]
    .apply(add_data1_share)
)
display(applied)

In [ ]:
external_key = [0, 1, 0, 1, 2, 0]
display(group_data.groupby(external_key)[["data1", "data2"]].sum())

indexed = group_data.set_index("key")
mapping = {"A": "vowel", "B": "consonant", "C": "consonant"}
display(indexed.groupby(mapping)[["data1", "data2"]].mean())
display(indexed.groupby([str.lower, mapping])[["data1", "data2"]].mean())

In [ ]:
planets = pd.read_csv(DATA_DIR / "planets.csv")
print(f"Planets: {planets.shape}")
display(planets.head())

median_period = planets.groupby("method")["orbital_period"].median().sort_values()
display(median_period)

decade = (10 * (planets["year"] // 10)).astype("Int64").astype(str) + "s"
decade.name = "decade"
discoveries = (
    planets.groupby(["method", decade], observed=False)["number"]
    .sum().unstack(fill_value=0)
)
display(discoveries)

In [ ]:
assert centered.shape == group_data[["data1", "data2"]].shape
assert np.allclose(centered.groupby(group_data["key"])[["data1", "data2"]].mean(), 0)
assert planets.shape == (1035, 6)
assert discoveries.loc["Transit", "2010s"] == 712
print("✓ Module 23 đạt các kiểm tra cơ bản.")

## 4. Pivot Table

`pivot_table()` là cách viết gọn cho chuỗi thao tác `groupby()` nhiều khóa → tổng hợp → `unstack()`. Ba thành phần cần đọc theo câu hỏi phân tích: `values` là đại lượng cần tính, `index/columns` là chiều so sánh, `aggfunc` là phép tổng hợp.

In [ ]:
titanic = pd.read_csv(DATA_DIR / "titanic.csv")
print(f"Titanic: {titanic.shape}")
print(f"Số dòng trùng hoàn toàn: {titanic.duplicated().sum()}")
display(titanic.isna().sum().sort_values(ascending=False).to_frame("missing"))
display(titanic.head())

In [ ]:
groupby_result = (
    titanic.groupby(["sex", "class"], observed=False)["survived"]
    .mean().unstack()
)
pivot_result = titanic.pivot_table(
    values="survived", index="sex", columns="class",
    aggfunc="mean", observed=False,
)
display(pivot_result)
pd.testing.assert_frame_equal(groupby_result, pivot_result)

In [ ]:
age_group_titanic = pd.cut(
    titanic["age"], bins=[0, 18, 80], labels=["0-18", "19-80"], include_lowest=True,
)
multi_level_pivot = titanic.pivot_table(
    values="survived", index=["sex", age_group_titanic], columns="class",
    aggfunc="mean", observed=False,
)
display(multi_level_pivot)

pivot_with_margins = titanic.pivot_table(
    values="survived", index="sex", columns="class",
    aggfunc="mean", margins=True, observed=False,
)
display(pivot_with_margins)

`margins=True` thêm hàng/cột `All`, nhưng `All` được tính lại từ dữ liệu gốc chứ không đơn giản là trung bình các ô đang thấy. Điều này quan trọng khi kích thước các nhóm khác nhau.

In [ ]:
births = pd.read_csv(DATA_DIR / "births.csv")
births["decade"] = 10 * (births["year"] // 10)
births_by_decade = births.pivot_table(
    values="births", index="decade", columns="gender",
    aggfunc="sum", observed=False,
)
print(f"CDC Births: {births.shape}")
display(births_by_decade)

In [ ]:
assert titanic.shape == (891, 15)
assert titanic["age"].isna().sum() == 177
assert np.isclose(pivot_with_margins.loc["All", "All"], titanic["survived"].mean())
assert births.shape[0] == 15547
assert births_by_decade.loc[2000, "M"] == 19106428
print("✓ Module 24 đạt các kiểm tra cơ bản.")

## 5. Case study Titanic - phải đọc `mean` cùng `count`

Tỉ lệ sống sót là trung bình của cột nhị phân `survived`. Tuy nhiên, một tỉ lệ rất cao từ nhóm chỉ có vài hành khách không đáng tin cậy bằng tỉ lệ gần tương đương từ nhóm hàng trăm người. Vì vậy, luôn xem đồng thời **mức độ** (`mean`) và **cỡ mẫu** (`count`).

In [ ]:
fare_quartile = pd.qcut(titanic["fare"], q=4, duplicates="drop")
fare_summary = titanic.pivot_table(
    values="survived", index=fare_quartile, columns="class",
    aggfunc=["mean", "count"], observed=False, fill_value=0,
)
display(fare_summary)

port_summary = (
    titanic.groupby(["embark_town", "class"], observed=False)["survived"]
    .agg(mean="mean", count="count")
    .dropna(subset=["mean"])
    .sort_values(["mean", "count"], ascending=[False, False])
)
display(port_summary)

## 6. Case study MovieLens 1M

Cell dưới đây chỉ chạy khi có đủ `movies.dat`, `users.dat`, `ratings.dat` và file ratings có đúng **1.000.209 dòng**. Kiểm tra này ngăn kết luận sai khi file lớn bị tải dở. Nếu chưa có, tải bộ MovieLens 1M từ GroupLens và đặt ba file vào `datasets/buoi6/movielens/`.

In [ ]:
users_path = MOVIELENS_DIR / "users.dat"
ratings_path = MOVIELENS_DIR / "ratings.dat"
required_paths = [movies_path, users_path, ratings_path]

if not all(path.exists() for path in required_paths):
    missing = [path.name for path in required_paths if not path.exists()]
    print(f"Bỏ qua case study MovieLens: còn thiếu {missing}.")
else:
    users = pd.read_table(
        users_path, sep="::", header=None,
        names=["user_id", "gender", "age", "occupation", "zip"],
        engine="python", encoding="latin-1",
    )
    ratings = pd.read_table(
        ratings_path, sep="::", header=None,
        names=["user_id", "movie_id", "rating", "timestamp"],
        engine="python", encoding="latin-1",
    )
    assert len(ratings) == 1_000_209, "ratings.dat không đầy đủ; không được phân tích file tải dở."
    assert len(users) == 6_040 and len(movies) == 3_883

    movie_data = (
        ratings.merge(users, on="user_id", validate="many_to_one")
        .merge(movies, on="movie_id", validate="many_to_one")
    )
    mean_ratings = movie_data.pivot_table(
        values="rating", index="title", columns="gender", aggfunc="mean", observed=False,
    )
    ratings_by_title = movie_data.groupby("title").size()
    active_titles = ratings_by_title.index[ratings_by_title >= 250]
    active_means = mean_ratings.loc[active_titles].dropna(subset=["F", "M"]).copy()
    active_means["diff_M_minus_F"] = active_means["M"] - active_means["F"]

    print("Ba phim được nữ giới đánh giá cao nhất:")
    display(active_means.sort_values("F", ascending=False).head(3))
    print("Năm phim được nam giới đánh giá cao nhất:")
    display(active_means.sort_values("M", ascending=False).head(5))
    print("Phim nữ giới thích hơn nam giới rõ nhất (M - F nhỏ nhất):")
    display(active_means.sort_values("diff_M_minus_F").head(5))

## 7. Bài tự luyện

1. Titanic: nhóm nào có `mean` cao nhưng `count` quá nhỏ để kết luận chắc chắn?
2. Titanic: thay `qcut(fare, 4)` bằng `cut()` với ranh giới nghiệp vụ và so sánh kết quả.
3. Planets: dùng `agg()` để tính đồng thời median, min, max của `orbital_period` theo `method`.
4. MovieLens: nếu đủ dữ liệu, dùng `age` và `gender` để lập pivot table rating trung bình; sau đó tính rating trung bình theo từng thể loại multi-label.

> Khi làm bài, hãy viết thêm ít nhất một `assert` kiểm tra shape, miền giá trị hoặc tính đầy đủ của kết quả.

In [ ]:
# Gợi ý khung làm bài 3 - hãy thay TODO bằng phần phân tích của bạn.
planet_exercise = planets.groupby("method")["orbital_period"].agg(["median", "min", "max"])
display(planet_exercise.head())
assert set(planet_exercise.columns) == {"median", "min", "max"}

In [ ]:
# Kiểm tra tích hợp cuối notebook
assert DATA_DIR.exists()
assert not any(frame.empty for frame in [movies, planets, titanic, births])
assert titanic["survived"].between(0, 1).all()
assert pivot_result.index.is_unique and pivot_result.columns.is_unique
print("✓ Notebook Buổi 6 chạy tuần tự và các kết quả cốt lõi hợp lệ.")

## 8. Ba kiến thức cần nhớ

1. **Làm sạch phải dựa trên ngữ nghĩa:** khóa nghiệp vụ quyết định dòng nào là trùng; ngữ cảnh quyết định sentinel hay outlier có thật sự là lỗi.
2. **Chọn đúng phép GroupBy:** `agg` để thu gọn, `filter` để loại nhóm, `transform` để giữ số dòng, `apply` chỉ khi cần logic tùy biến.
3. **Không đọc tỉ lệ mà bỏ qua cỡ mẫu:** trong pivot table, nên xem `mean` cùng `count`; `margins` được tính từ dữ liệu gốc và có trọng số theo số quan sát.

## Tài liệu đối chiếu

- Bài giảng `slides/buoi6_python_datascience.pdf`.
- *Python Data Science Handbook*: notebook 03.08 Aggregation and Grouping, 03.09 Pivot Tables.
- *Python for Data Analysis*: phần Data Cleaning and Preparation và MovieLens.
- GroupLens MovieLens 1M; `mwaskom/seaborn-data`; CDC Births.